# 01 - Preprocessing

Adapted from the Urban Taxonomy / HiMoC pipeline for **Winnipeg, Canada**.

- Buildings source: OBPL-CA-v3 footprints **enriched with CHM canopy stats** (chm_mean, chm_std, chm_max, chm_cover)
- Street network: OSM roads via GeoPackage
- Canadian additions: HEIGHT, volume, floor_count from OBPL; canopy stats from LiDAR CHM

Steps:
1. Load buildings with pre-computed CHM stats, reproject to analysis CRS, clip to city boundary
2. Clean buildings: make_valid, explode, simplify, merge overlaps
3. Load roads, filter by approved classes, reproject
4. Fix nulls in HEIGHT and CHM columns
5. Save preprocessed data

In [1]:
import os
# Navigate to project root from any notebooks_* subdirectory
if os.path.basename(os.getcwd()).startswith("notebooks"):
    os.chdir(os.path.dirname(os.getcwd()))
print("Working dir:", os.getcwd())

import sys
sys.path.insert(0, "scripts")
from pipeline_config import load_city_config
load_city_config()

Working dir: C:\urban-form\urban_taxonomy
[pipeline_config] City config: config\canada_cma_ca_nochm.env


WindowsPath('config/canada_cma_ca_nochm.env')

In [2]:
import geopandas as gpd
import numpy as np
import pandas as pd
import momepy as mm
import geoplanar
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [3]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# All values below come from the city config file (config/<city>.env, selected
# via the CITY_CONFIG env var — see scripts/pipeline_config.py) so switching
# cities means picking a different config file, not editing this notebook.
# The literal defaults here match Winnipeg and only kick in if a value is
# missing from both the environment and the config file.
# ──────────────────────────────────────────────────────────────────────────────

CITY_NAME = os.environ.get("CITY_NAME", "Winnipeg")  # used in print messages only
CITY_SLUG = os.environ.get("CITY_SLUG", CITY_NAME.lower())  # used in export filenames

# Projected CRS for all spatial analysis (must be in metres).
TARGET_CRS = os.environ.get("TARGET_CRS", "EPSG:3978")

# --- Input data paths (relative to project root) ---
BUILDINGS_FILE  = os.environ.get("BUILDINGS_FILE", "data/winnipeg_buildings_chm_stats.gpkg")
BUILDINGS_LAYER = os.environ.get("BUILDINGS_LAYER", "buildings_chm_stats")  # GeoPackage layer name
BOUNDARY_FILE   = os.environ.get("BOUNDARY_FILE", "data/winnipeg_boundry.gpkg")
ROADS_FILE      = os.environ.get("ROADS_FILE", "data/winnipeg_roads.gpkg")
# Column holding the OSM road classification (residential/primary/etc). Name
# varies by extract/tool — some use "fclass", others "highway". If the
# configured column isn't found, the roads-loading cell below falls back to
# the other common name before failing loudly.
ROAD_CLASS_COL  = os.environ.get("ROAD_CLASS_COL", "fclass")

# --- CHM (canopy height model) toggle ---
# Set INCLUDE_CHM=0 in the city config to exclude CHM canopy attributes from
# the whole pipeline (e.g. no LiDAR coverage for this city, or to compare
# taxonomy with/without tree canopy features). When False, the CHM columns
# are dropped right after loading (below) and every downstream notebook
# skips them automatically since they check for column presence before use.
INCLUDE_CHM = os.environ.get("INCLUDE_CHM", "1").strip().lower() not in ("0", "false", "no")

# --- CHM attribute column names (must match fields in BUILDINGS_FILE) ---
# Computed externally: buffer each building, then sample arcgis/merge_raster.tif.
#   chm_mean  – mean canopy height (m) within buffer
#   chm_std   – std dev of height — captures canopy structural complexity
#   chm_max   – peak canopy height (m) within buffer
#   chm_cover – fraction of buffer pixels with height ≥ 2 m (canopy presence)
CHM_COLS = [c.strip() for c in os.environ.get("CHM_COLS", "chm_mean,chm_std,chm_max,chm_cover").split(",") if c.strip()]

# --- Geometry cleaning parameters (source: generate_buildings.py) ---
SIMPLIFICATION_TOLERANCE = float(os.environ.get("SIMPLIFICATION_TOLERANCE", "0.1"))    # metres
MERGE_LIMIT              = float(os.environ.get("MERGE_LIMIT", "25"))                  # m², min overlap to trigger merge
MAX_BUILDING_AREA        = float(os.environ.get("MAX_BUILDING_AREA", "200000"))        # m², drop footprints larger than this

# --- Output ---
from pathlib import Path
OUTPUT_ROOT = os.environ.get("OUTPUT_ROOT", "outputs")  # root folder for all pipeline outputs; vary per city to avoid clobbering
OUT = Path(OUTPUT_ROOT) / "01_preprocessed"
OUT.mkdir(parents=True, exist_ok=True)

print(f"City: {CITY_NAME} ({CITY_SLUG})  |  INCLUDE_CHM={INCLUDE_CHM}  |  OUTPUT_ROOT={OUTPUT_ROOT}")

City: Canada (canada_cma_ca_995)  |  INCLUDE_CHM=False  |  OUTPUT_ROOT=outputs_canada_cma_ca_nochm\regions\995


## 1. Winnipeg boundary

In [4]:
boundary = gpd.read_file(BOUNDARY_FILE).to_crs(TARGET_CRS)
boundary_union = boundary.union_all()
print("Boundary loaded, CRS:", boundary.crs)

Boundary loaded, CRS: EPSG:3979


## 2. Load and clean buildings

Following the source `generate_buildings.py` pipeline:
1. `make_valid()` 
2. Explode MultiPolygons, keep only Polygons
3. Simplify geometry (0.1 m tolerance)
4. Drop buildings > 200,000 m^2
5. `geoplanar.merge_overlaps()` — merge buildings overlapping >10% or < 25 m^2
6. `geoplanar.trim_overlaps()` — remove remaining overlaps
7. Merge touching collapsing buildings
8. Final simplify + make_valid

In [5]:
import pyogrio

# Discover available layers so we fail with a clear message if BUILDINGS_LAYER is wrong
available_layers = [row[0] for row in pyogrio.list_layers(BUILDINGS_FILE)]
print(f"Available layers: {available_layers}")

layer = BUILDINGS_LAYER if BUILDINGS_LAYER in available_layers else available_layers[0]
if layer != BUILDINGS_LAYER:
    print(f"  WARNING: '{BUILDINGS_LAYER}' not found — using '{layer}'. Update BUILDINGS_LAYER in config.")

# Load buildings enriched with CHM canopy stats.
# Contains all OBPL attributes (HEIGHT, BLDG_POP, …) plus chm_mean/std/max/cover.
# `bbox=boundary` lets pyogrio use the GeoPackage's spatial index to only pull
# features whose bbox intersects our boundary (geopandas reprojects the bbox
# to the file's native CRS automatically) — for a single-city run this is a
# free speedup; for a region-chunked run (scripts/region_pipeline.py) it's
# what makes reading BUILDINGS_FILE 100+ times per country feasible at all,
# instead of loading all of it fresh for every region.
buildings = gpd.read_file(BUILDINGS_FILE, layer=layer, bbox=boundary)

# Source file is in EPSG:3979 (NAD83 CSRS Lambert); reproject to analysis CRS
buildings = buildings.to_crs(TARGET_CRS)

# Clip to city boundary (bbox above is a coarse prefilter; this is the exact clip)
buildings = buildings[buildings.intersects(boundary_union)].copy()
buildings = buildings.reset_index(drop=True)
initial_count = len(buildings)
print(f"Loaded {initial_count:,} buildings within {CITY_NAME}")

Available layers: ['buildings_chm_stats']


Loaded 6,684 buildings within Canada

In [6]:
# --- Source pipeline: process_region_buildings() ---
# Cleaning parameters are defined in the Configuration cell above.

# 1. Fix invalid geometry
buildings["geometry"] = buildings.make_valid()

# 2. Explode + keep only Polygons
buildings = buildings.explode(ignore_index=True)
buildings = buildings[buildings.geom_type == "Polygon"].reset_index(drop=True)

# 3. Simplify
buildings["geometry"] = buildings.simplify(SIMPLIFICATION_TOLERANCE).normalize()

# 4. Drop very large footprints
buildings = buildings[buildings.area < MAX_BUILDING_AREA].reset_index(drop=True)

# 5. Merge overlapping buildings
buildings = geoplanar.merge_overlaps(buildings, merge_limit=MERGE_LIMIT, overlap_limit=0.1)

# 6. Trim remaining overlaps
buildings = geoplanar.trim_overlaps(buildings, strategy="smallest")

# 7. Explode again + keep Polygons
buildings = buildings.explode(ignore_index=True)
buildings = buildings[buildings.geom_type == "Polygon"].reset_index(drop=True)

# 8. Merge touching collapsing buildings (shrink test)
shrink = buildings.buffer(-0.5, resolution=2)
buildings = geoplanar.merge_touching(buildings, np.where(shrink.is_empty), largest=True)
buildings = buildings.explode(ignore_index=True)
buildings = buildings[buildings.geom_type == "Polygon"].reset_index(drop=True)

# 9. Final simplify + make_valid
buildings["geometry"] = buildings.simplify(SIMPLIFICATION_TOLERANCE)
buildings["geometry"] = buildings.make_valid()
buildings = buildings[buildings.geom_type == "Polygon"].reset_index(drop=True)
buildings["geometry"] = buildings.normalize()

print(f"After cleaning: {len(buildings):,} buildings (dropped {1 - len(buildings)/initial_count:.1%})")

After cleaning: 6,668 buildings (dropped 0.2%)


In [7]:
# Fix nulls in HEIGHT — used for volume computation in notebook 04
buildings["HEIGHT"] = buildings["HEIGHT"].fillna(0).clip(lower=0)

if INCLUDE_CHM:
    # Fill nulls in CHM canopy stats — buildings with no LiDAR coverage get 0
    # (0 = no canopy: bare ground, paved surfaces, or outside raster extent)
    for col in CHM_COLS:
        if col in buildings.columns:
            buildings[col] = buildings[col].fillna(0).clip(lower=0)
else:
    # CHM disabled for this run — drop the canopy columns so the rest of the
    # pipeline (which checks `if col in buildings.columns`) skips them cleanly.
    buildings = buildings.drop(columns=[c for c in CHM_COLS if c in buildings.columns])

print(f"CHM canopy attributes: {'included' if INCLUDE_CHM else 'excluded'}")
print(f"Columns: {list(buildings.columns)}")

CHM canopy attributes: excluded
Columns: ['BLDG_ID', 'chm_edge_flag', 'city_name', 'chm_processed_date', 'CMAUID', 'CMANAME', 'CMATYPE', 'HEIGHT', 'DBUID', 'DB_RATIO', 'DAUID', 'DA_RATIO', 'CSDUID', 'CSD_RATIO', 'CDUID', 'CD_RATIO', 'ORPHAN_RATIO', 'HAS_ORPHAN_BONUS', 'BLDG_POP', 'POP_SOURCE', 'QF_O', 'QF_V', 'geometry']


## 3. Load and filter roads

Source `generate_streets.py` uses this approved list (service is **excluded**):
```
living_street, motorway, motorway_link, pedestrian, primary, primary_link,
residential, secondary, secondary_link, tertiary, tertiary_link,
trunk, trunk_link, unclassified
```

We follow the same list. Source also runs `neatnet.neatify()` for topology
cleanup but that requires specific Overture schema; we skip it for OSM data.

In [8]:
roads = gpd.read_file(ROADS_FILE, bbox=boundary).to_crs(TARGET_CRS)

# Source approved_roads list (service excluded)
APPROVED_ROADS = {
    "living_street", "motorway", "motorway_link", "pedestrian",
    "primary", "primary_link", "residential",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link",
    "trunk", "trunk_link", "unclassified",
}

# --- Road class column check ---
# Different OSM extracts/tools name this column differently ("fclass" vs
# "highway"). Fall back to the other common name if the configured one is
# missing, then fail with a clear message (listing available columns) rather
# than a raw KeyError deep in a filter expression.
road_class_col = ROAD_CLASS_COL
if road_class_col not in roads.columns:
    fallback = next((c for c in ("fclass", "highway") if c in roads.columns), None)
    if fallback is None:
        raise KeyError(
            f"Road class column '{road_class_col}' not found in {ROADS_FILE}. "
            f"Available columns: {list(roads.columns)}. "
            f"Set ROAD_CLASS_COL in the city config to the correct column name."
        )
    print(f"  WARNING: ROAD_CLASS_COL='{road_class_col}' not found — using '{fallback}'. Update ROAD_CLASS_COL in config.")
    road_class_col = fallback

roads = roads[roads[road_class_col].isin(APPROVED_ROADS)].copy()
roads = roads[roads.intersects(boundary_union)].copy()
roads = roads.explode(index_parts=False).reset_index(drop=True)

# Sort by a stable ID column if present, else fall back to positional order
sort_col = next((c for c in ("osm_id", "id") if c in roads.columns), None)
if sort_col:
    roads = roads.sort_values(sort_col).reset_index(drop=True)
else:
    roads = roads.reset_index(drop=True)

print(f"{len(roads):,} road segments")
print(roads[road_class_col].value_counts().to_string())

716 road segments
highway
residential      502
tertiary         113
unclassified      23
secondary         22
trunk             21
pedestrian        14
primary            9
tertiary_link      7
trunk_link         5


## 4. Save preprocessed data

In [9]:
buildings.to_file(OUT / "buildings.gpkg", driver="GPKG")
roads.to_file(OUT / "roads.gpkg", driver="GPKG")
boundary.to_file(OUT / "boundary.gpkg", driver="GPKG")

print(f"Saved to {OUT}")
print(f"  buildings.gpkg : {len(buildings):,} buildings")
print(f"  roads.gpkg     : {len(roads):,} road segments")
print(f"  boundary.gpkg  : 1 polygon")

Saved to outputs_canada_cma_ca_nochm\regions\995\01_preprocessed


  buildings.gpkg : 6,668 buildings
  roads.gpkg     : 716 road segments
  boundary.gpkg  : 1 polygon
